# Lab 2: Pre-trained CNNs, Transfer Learning, and Fine-Tuning on CIFAR-10
## Báo Cáo Thực Hành Lab 2: Mạng CNN Tiền Huấn Luyện, Học Chuyển Giao và Tinh Chỉnh Trên CIFAR-10

**Mục tiêu bài thực hành (Lab Goals):**
1. Tái sử dụng các mạng CNN hiện đại được tiền huấn luyện trên tập ImageNet (`ResNet18`, `DenseNet121`, `VGG16`) từ `torchvision.models` cho bài toán phân loại ảnh 10 lớp trên CIFAR-10.
2. Thực hiện kỹ thuật **Học chuyển giao (Transfer Learning)**: Thay thế lớp phân loại (head/classifier) cuối cùng, đóng băng toàn bộ backbone (`freeze_backbone=True`).
3. Thực hiện kỹ thuật **Tinh chỉnh (Fine-Tuning)**: Mở khóa các khối tích chập cuối cùng (`unfreeze_last_blocks`) với tốc độ học phù hợp để thích ứng sâu hơn với phân bố dữ liệu mới.
4. Chuẩn hóa dữ liệu theo chuẩn ImageNet (`IMAGENET_MEAN`, `IMAGENET_STD`) và áp dụng tăng cường dữ liệu (Data Augmentation) hợp lý trên tập huấn luyện.
5. Đánh giá toàn diện mô hình: Theo dõi quá trình huấn luyện bằng TensorBoard, phân tích ma trận nhầm lẫn (Confusion Matrix), phát hiện các cặp lớp dễ nhầm lẫn nhất (Top Confusions).
6. Thử nghiệm so sánh có đối chứng giữa các kiến trúc mạng và các bộ siêu tham số (Hyperparameter Grid Search).

```mermaid
flowchart TD
    A["Dữ liệu thô CIFAR-10<br/>(50k Train, 10k Test)"] --> B["1. Khám phá dữ liệu - EDA<br/>(Phân bố lớp & Mean/Std)"]
    B --> C["2. Tiền xử lý & Split 90/10<br/>(45k Train, 5k Val - Cố định Seed)"]
    C --> D["3. Features & DataLoaders<br/>(Resize, Augment, ImageNet Normalization)"]
    D --> E["4. Khởi tạo Mô hình CNN<br/>(Thay thế lớp cuối: fc/classifier -> 10 classes)"]
    E --> F["5. Huấn luyện & Checkpoint<br/>(Loss, Optimizer, TensorBoard Logging)"]
    F --> G["Checkpoint tốt nhất: best.pt"]
    G --> H["6. Đánh giá Validation & Ma trận nhầm lẫn"]
    G --> I["7. Phân tích lỗi & Top Confusions"]
    G --> J["8. Thử nghiệm so sánh đa mô hình"]
    J --> K["9. Đánh giá cuối cùng trên tập Test"]
    K --> L["10. Tổng kết & Vấn đáp bảo vệ"]
```

### Bảng Tổng Hợp Các Bước Trong Pipeline
| Bước | Module | Các hàm chính | Sản phẩm đầu ra (Artifacts) |
|:---:|:---|:---|:---|
| **1** | `eda` | `run_eda` | Số lượng mẫu, `cifar10_sample_grid.png`, `cifar10_class_distribution.png` |
| **2** | `preprocessing` | `load_raw_dataset`, `make_split`, `save_split`, `load_split` | Tập train/val không rò rỉ, `train_val_split.json` |
| **3** | `features` | `build_transforms`, `make_loaders` | DataLoaders chuẩn hóa ImageNet `[B, 3, H, W]` |
| **4** | `model` | `build_model`, `describe_model`, `unfreeze_last_blocks` | Mô hình PyTorch xuất logits `[B, 10]`, thống kê tham số |
| **5** | `train` | `train_model`, `save_checkpoint`, `load_checkpoint` | Checkpoint `best.pt`, log TensorBoard, biểu đồ Loss/Acc |
| **6** | `evaluate` | `evaluate`, `confusion_matrix`, `top_confusions` | Độ chính xác val, ma trận nhầm lẫn 10x10, Top 5 cặp lỗi |
| **7** | `experiment` | `run_comparison`, `save_results` | Bảng so sánh đa cấu hình `model_comparison.csv` |
| **8** | `evaluate` | `evaluate` (trên test_loader) | Độ chính xác và độ mất mát trên 10,000 ảnh test độc lập |

## 1. Thiết lập Môi trường & Nạp Thư viện (Setup & Environment)
**Mục đích:** 
- Tự động định vị thư mục gốc của repository (chứa cả `shared/` và `labs/`) và thêm vào `sys.path` để đảm bảo code chạy được trên mọi môi trường mà không bị lỗi đường dẫn tương đối.
- Cố định hạt giống ngẫu nhiên (seed) bằng hàm `set_seed` từ `shared.seed` để tái lập hoàn toàn kết quả.
- Tự động nhận diện thiết bị tính toán phần cứng (`cuda` nếu có GPU, ngược lại dùng `cpu`).
- Nạp đầy đủ tất cả các hàm từ các module: `eda`, `preprocessing`, `features`, `model`, `train`, `evaluate`, `experiment`.

In [5]:
import sys
import os
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# 1. Tự động xác định thư mục gốc của repository (chứa thư mục 'shared' và 'labs')
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "shared").is_dir() and (p / "labs").is_dir()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 2. Nạp các module quản lý đường dẫn và seed dùng chung
from shared.paths import raw_data_dir, artifact_dir
from shared.seed import set_seed

# 3. Nạp đầy đủ các hàm từ từng module thành phần của Lab 2
from labs.lab2.eda.eda import run_eda, CIFAR10_CLASSES
from labs.lab2.preprocessing.preprocessing import load_raw_dataset, make_split, save_split, load_split
from labs.lab2.features.features import (
    build_transforms, make_loaders, TransformedSubset, TransformedDataset,
    IMAGENET_MEAN, IMAGENET_STD
)
from labs.lab2.model.model import build_model, describe_model, unfreeze_last_blocks
from labs.lab2.train.train import train_model, save_checkpoint, load_checkpoint
from labs.lab2.evaluate.evaluate import evaluate, confusion_matrix, top_confusions
from labs.lab2.experiment.experiment import run_comparison, save_results

# 4. Cố định hạt giống ngẫu nhiên (seed=42) để tái lập kết quả thử nghiệm
SEED = 42
set_seed(SEED)

# 5. Lựa chọn thiết bị tính toán phần cứng
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Thư mục gốc của repository (repo_root): {repo_root}")
print(f"Thiết bị tính toán được chọn (device)   : {device}")
print(f"Danh sách 10 lớp CIFAR-10               : {CIFAR10_CLASSES}")

ModuleNotFoundError: No module named 'shared'

## 2. Khám phá Dữ liệu - EDA (`labs.lab2.eda`)
**Mục đích:**
- Gọi hàm `run_eda(data_root, out_dir)` để khám phá dữ liệu CIFAR-10 ở trạng thái thô (chỉ đọc, không làm biến đổi dữ liệu).
- Xác định tổng số mẫu huấn luyện (50,000) và kiểm thử (10,000).
- Kiểm tra tính cân bằng giữa các lớp: CIFAR-10 có đúng 5,000 mẫu/lớp trong tập train.
- Tính toán giá trị trung bình (`channel_mean`) và độ lệch chuẩn (`channel_std`) của các kênh màu RGB thô.
- Lưu và hiển thị trực quan 2 biểu đồ: Lưới ảnh mẫu đại diện 10 lớp (`cifar10_sample_grid.png`) và biểu đồ cột phân bố lớp (`cifar10_class_distribution.png`).

In [ ]:
# Đường dẫn thư mục dữ liệu thô và thư mục lưu biểu đồ kết quả
data_root = str(raw_data_dir("lab2"))
figures_dir = str(artifact_dir("lab2", "figures"))

print(f"[EDA] Tiến hành phân tích tập dữ liệu CIFAR-10 tại: {data_root}")
eda_stats = run_eda(data_root=data_root, out_dir=figures_dir)

# Hiển thị các chỉ số thống kê cơ bản thu được từ EDA
print("\n--- THỐNG KÊ TẬP DỮ LIỆU CIFAR-10 ---")
print(f"+ Số lượng ảnh tập Train: {eda_stats['n_train']:,} ảnh")
print(f"+ Số lượng ảnh tập Test : {eda_stats['n_test']:,} ảnh")
print(f"+ Kích thước ảnh gốc    : {eda_stats['image_size']} (Cao x Rộng x Kênh màu)")
print(f"+ Giá trị Mean các kênh (RGB): {eda_stats['channel_mean']}")
print(f"+ Giá trị Std các kênh (RGB) : {eda_stats['channel_std']}")

print("\n+ Phân bố số lượng mẫu theo từng lớp:")
for cls_name, count in eda_stats["class_counts"].items():
    print(f"  - Lớp {cls_name:12s}: {count:,} mẫu")

In [ ]:
# Hiển thị trực quan 2 biểu đồ do module EDA tạo ra trong artifacts/figures/
sample_grid_path = Path(figures_dir) / "cifar10_sample_grid.png"
class_dist_path = Path(figures_dir) / "cifar10_class_distribution.png"

if sample_grid_path.is_file():
    print("Lưới ảnh mẫu đại diện (3 mẫu cho mỗi lớp trong 10 lớp):")
    display(Image(filename=str(sample_grid_path)))

if class_dist_path.is_file():
    print("Biểu đồ phân bố 10 lớp (Phân bố hoàn toàn đồng đều 5,000 mẫu/lớp):")
    display(Image(filename=str(class_dist_path)))

## 3. Tiền Xử Lý & Phân Chia Tập Dữ Liệu (`labs.lab2.preprocessing`)
**Mục đích:**
- `load_raw_dataset(data_root, download=True)`: Nạp dataset CIFAR-10 gốc chưa qua bất kỳ bước transform nào (giữ nguyên dạng ảnh PIL) để các bước sau có thể áp dụng transform độc lập.
- `make_split(n_samples, val_ratio=0.1, seed=42)`: Tạo phân chia ngẫu nhiên nhưng có thể tái lập chính xác (reproducible) thành 90% tập Train (45,000 mẫu) và 10% tập Validation (5,000 mẫu).
- `save_split` & `load_split`: Lưu lại các chỉ số (indices) vào file JSON tại `artifacts/splits/train_val_split.json` và nạp lại. Điều này đảm bảo toàn bộ nhóm sử dụng chung một tập train/val cố định, tránh hiện tượng rò rỉ dữ liệu (data leakage) giữa các lần chạy.

In [ ]:
# 1. Nạp dataset CIFAR-10 thô
train_set_raw, test_set_raw = load_raw_dataset(data_root=data_root, download=True)
print(f"Nạp thành công dataset thô: {len(train_set_raw):,} mẫu train, {len(test_set_raw):,} mẫu test.")

# 2. Đường dẫn lưu trữ phân chia tập Train / Validation
split_path = str(artifact_dir("lab2", "splits") / "train_val_split.json")

# Kiểm tra nếu đã tồn tại file split JSON thì nạp lại, nếu chưa thì tạo mới và lưu lại
if Path(split_path).is_file():
    print(f"Nạp lại chỉ số phân chia từ file có sẵn: {split_path}")
    train_idx, val_idx = load_split(split_path)
else:
    print(f"Chưa có file split. Đang tạo phân chia mới 90/10 với seed={SEED}...")
    train_idx, val_idx = make_split(n_samples=len(train_set_raw), val_ratio=0.1, seed=SEED)
    save_split(train_idx, val_idx, split_path)
    print(f"Đã lưu thành công split vào: {split_path}")

print(f"+ Kích thước tập Huấn luyện (Train) : {len(train_idx):,} mẫu ({len(train_idx)/len(train_set_raw)*100:.1f}%)")
print(f"+ Kích thước tập Đánh giá (Val)    : {len(val_idx):,} mẫu ({len(val_idx)/len(train_set_raw)*100:.1f}%)")

# 3. Xác thực không có sự giao thoa chỉ số giữa 2 tập
intersection = set(train_idx).intersection(set(val_idx))
assert len(intersection) == 0, f"LỖI: Phát hiện {len(intersection)} mẫu bị trùng lặp giữa Train và Val!"
print("Xác thực hợp lệ: Không có mẫu trùng lặp giữa tập Train và Validation (0 data leakage).")

## 4. Xây dựng Biến Đổi Đặc Trưng & DataLoaders (`labs.lab2.features`)
**Mục đích:**
- `build_transforms(image_size, augment)`: 
  - Chuẩn hóa màu sắc bằng `IMAGENET_MEAN = [0.485, 0.456, 0.406]` và `IMAGENET_STD = [0.229, 0.224, 0.225]`. Đây là yêu cầu bắt buộc khi sử dụng backbone đã pre-trained trên ImageNet.
  - Resize ảnh từ kích thước gốc 32x32 lên kích thước phù hợp (`image_size=128` hoặc `224`). Sử dụng 128x128 giúp cân bằng tối ưu giữa tốc độ huấn luyện trên CPU/GPU cá nhân và độ chi tiết đặc trưng của CNN.
  - Tập **Train**: Áp dụng Data Augmentation nhẹ (`RandomHorizontalFlip`, `RandomResizedCrop`) để chống overfitting.
  - Tập **Validation/Test**: Chỉ Resize + ToTensor + Normalize (tuyệt đối không áp dụng data augmentation ngẫu nhiên).
- `make_loaders`: Tạo các `DataLoader` với `num_workers=0` (đảm bảo tính ổn định và tránh lỗi đa tiến trình trên Windows).

In [ ]:
# Thiết lập kích thước ảnh và kích thước lô (batch size)
IMAGE_SIZE = 128  # Kích thước 128x128 cân bằng hoàn hảo giữa tốc độ và độ chính xác
BATCH_SIZE = 64

# 1. Xây dựng transform cho tập train (có augmentation) và eval (không augmentation)
train_tf, eval_tf = build_transforms(image_size=IMAGE_SIZE, augment=True)

print("Transform áp dụng cho tập Train:\n", train_tf)
print("\nTransform áp dụng cho tập Val/Test:\n", eval_tf)

# 2. Khởi tạo DataLoaders cho 3 tập: Train, Validation và Test
train_loader, val_loader, test_loader = make_loaders(
    train_set=train_set_raw,
    test_set=test_set_raw,
    train_idx=train_idx,
    val_idx=val_idx,
    train_tf=train_tf,
    eval_tf=eval_tf,
    batch_size=BATCH_SIZE
)

print(f"\n+ Số lượng batches trong Train Loader: {len(train_loader)} batches (batch_size={BATCH_SIZE})")
print(f"+ Số lượng batches trong Val Loader  : {len(val_loader)} batches")
print(f"+ Số lượng batches trong Test Loader : {len(test_loader)} batches")

# 3. Kiểm tra shape và kiểu dữ liệu của một batch thực tế
batch_images, batch_labels = next(iter(train_loader))
print(f"\nKiểm tra Shape của 1 mini-batch:")
print(f"- Images Tensor: {batch_images.shape} -> [Batch={batch_images.shape[0]}, Channels={batch_images.shape[1]}, Height={batch_images.shape[2]}, Width={batch_images.shape[3]}]")
print(f"- Labels Tensor: {batch_labels.shape} -> Dải nhãn: [{batch_labels.min().item()} đến {batch_labels.max().item()}]")

## 5. Khởi Tạo Mô Hình CNN Tiền Huấn Luyện (`labs.lab2.model`)
**Mục đích:**
- `build_model(name, num_classes=10, freeze_backbone, pretrained=True)`:
  - Nạp trọng số ImageNet hiện đại bằng cú pháp chuẩn `weights=...Weights.DEFAULT` (thay vì cú pháp cũ `pretrained=True` đã bị deprecated trong torchvision mới).
  - Thay thế lớp phân loại cuối cùng (ví dụ: `fc` trong ResNet, `classifier` trong DenseNet) để xuất ra đúng 10 logits tương ứng 10 lớp CIFAR-10.
- `describe_model(model)`: Đếm tổng số tham số (`total_params`), số tham số cần cập nhật gradient (`trainable_params`) và liệt kê tên các khối con trong mạng.
- `unfreeze_last_blocks(model, n_blocks)`: Mở khóa có chọn lọc các khối tích chập bậc cao (ví dụ: `layer4` của ResNet-18) để thực hiện chiến lược **Fine-Tuning**.

In [ ]:
# === 1. Mô hình ResNet-18 đóng băng Backbone (Feature Extractor) ===
print("=== 1. ResNet-18 (Feature Extraction - Toàn bộ Backbone đóng băng) ===")
resnet_feat = build_model(name="resnet18", num_classes=10, freeze_backbone=True, pretrained=True)
info_feat = describe_model(resnet_feat)
print(f"Tổng số tham số         : {info_feat['total_params']:,}")
print(f"Số tham số được huấn luyện: {info_feat['trainable_params']:,} ({info_feat['trainable_params']/info_feat['total_params']*100:.2f}%)")
print(f"Lớp phân loại cuối cùng : {resnet_feat.fc}")
print(f"Các khối chính trong mạng : {info_feat['layer_names']}\n")

# === 2. Mô hình ResNet-18 mở khóa layer4 (Fine-Tuning) ===
print("=== 2. ResNet-18 (Fine-Tuning - Mở khóa khối layer4 và lớp fc) ===")
resnet_fine = build_model(name="resnet18", num_classes=10, freeze_backbone=True, pretrained=True)
unfreeze_last_blocks(resnet_fine, n_blocks=1)  # Mở khóa 1 block cuối (layer4)
info_fine = describe_model(resnet_fine)
print(f"Tổng số tham số         : {info_fine['total_params']:,}")
print(f"Số tham số được huấn luyện: {info_fine['trainable_params']:,} ({info_fine['trainable_params']/info_fine['total_params']*100:.2f}%)\n")

# === 3. Mô hình DenseNet-121 đóng băng Backbone ===
print("=== 3. DenseNet-121 (Feature Extraction) ===")
densenet_feat = build_model(name="densenet121", num_classes=10, freeze_backbone=True, pretrained=True)
info_dense = describe_model(densenet_feat)
print(f"Tổng số tham số         : {info_dense['total_params']:,}")
print(f"Số tham số được huấn luyện: {info_dense['trainable_params']:,} ({info_dense['trainable_params']/info_dense['total_params']*100:.2f}%)")
print(f"Lớp classifier cuối cùng: {densenet_feat.classifier}\n")

# === 4. Kiểm tra Forward pass với tensor mẫu ===
sample_input = torch.randn(4, 3, IMAGE_SIZE, IMAGE_SIZE)
logits_out = resnet_feat(sample_input)
print(f"Kiểm tra Forward Pass: Input shape = {sample_input.shape} -> Output logits shape = {logits_out.shape}")
assert logits_out.shape == (4, 10), "Lỗi: Shape logits đầu ra không khớp [4, 10]!"

## 6. Huấn Luyện Mô Hình & Checkpointing (`labs.lab2.train`)
**Mục đích:**
- `train_model`:
  - Thực hiện vòng lặp huấn luyện tính hàm mất mát `CrossEntropyLoss`.
  - Chỉ cập nhật gradient cho các tham số có `requires_grad=True` bằng thuật toán tối ưu `Adam` (hoặc `SGD`) kết hợp trọng số suy giảm `weight_decay`.
  - Sau mỗi epoch, tiến hành đánh giá loss và accuracy trên tập validation.
  - Tự động ghi nhận các chỉ số scalar (Train Loss, Val Loss, Val Accuracy) lên **TensorBoard** thông qua `SummaryWriter`.
  - Lưu lại checkpoint có `val_loss` thấp nhất bằng `save_checkpoint` tại đường dẫn chỉ định.
- `load_checkpoint`: Tải lại trọng số tốt nhất vào mô hình để phục vụ đánh giá hoặc suy luận.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# Thiết lập siêu tham số cho mô hình cơ sở (Baseline)
EPOCHS = 3           # Số epochs huấn luyện mẫu (có thể nâng lên 5-10 khi có GPU)
LR = 1e-3            # Tốc độ học (Learning rate)
OPTIMIZER_NAME = "adam"
WEIGHT_DECAY = 1e-4

# Thiết lập đường dẫn lưu checkpoint và log TensorBoard
models_dir = artifact_dir("lab2", "models")
logs_dir = artifact_dir("lab2", "logs")
baseline_ckpt = str(models_dir / "resnet18_baseline_best.pt")
baseline_tb_log = str(logs_dir / "resnet18_baseline")

# Khởi tạo TensorBoard SummaryWriter
writer = SummaryWriter(log_dir=baseline_tb_log)

# Khởi tạo mô hình ResNet-18 (Feature extraction)
model_to_train = build_model("resnet18", num_classes=10, freeze_backbone=True, pretrained=True)

print(f"[Bắt đầu Huấn luyện Mô hình Cơ sở ResNet-18 trên {device}]")
print(f"- Epochs: {EPOCHS} | LR: {LR} | Optimizer: {OPTIMIZER_NAME} | Weight Decay: {WEIGHT_DECAY}")
print(f"- Checkpoint lưu tại : {baseline_ckpt}")
print(f"- TensorBoard log tại: {baseline_tb_log}")

# Tiến hành huấn luyện mô hình
history = train_model(
    model=model_to_train,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device,
    ckpt_path=baseline_ckpt,
    writer=writer,
    optimizer_name=OPTIMIZER_NAME,
    weight_decay=WEIGHT_DECAY
)
writer.close()

print("\n--- KẾT QUẢ HUẤN LUYỆN QUA TỪNG EPOCH ---")
for ep in range(len(history["train_loss"])):
    print(f"Epoch {ep + 1:2d}/{EPOCHS}: "
          f"Train Loss = {history['train_loss'][ep]:.4f} | "
          f"Val Loss = {history['val_loss'][ep]:.4f} | "
          f"Val Accuracy = {history['val_acc'][ep] * 100:.2f}%")

In [ ]:
# Vẽ biểu đồ đường cong học tập (Training & Validation Curves)
epochs_list = list(range(1, len(history["train_loss"]) + 1))

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ Loss
ax_loss.plot(epochs_list, history["train_loss"], marker="o", color="#1f77b4", linewidth=2, label="Train Loss")
ax_loss.plot(epochs_list, history["val_loss"], marker="s", color="#ff7f0e", linewidth=2, label="Validation Loss")
ax_loss.set_title("Đồ Thị Hàm Mất Mát (Loss Curves)", fontsize=12, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=11)
ax_loss.set_ylabel("Cross Entropy Loss", fontsize=11)
ax_loss.set_xticks(epochs_list)
ax_loss.grid(True, linestyle="--", alpha=0.6)
ax_loss.legend(fontsize=11)

# Biểu đồ Accuracy
val_acc_pct = [acc * 100 for acc in history["val_acc"]]
ax_acc.plot(epochs_list, val_acc_pct, marker="^", color="#2ca02c", linewidth=2, label="Val Accuracy (%)")
ax_acc.set_title("Độ Chính Xác Trên Tập Validation (Val Accuracy)", fontsize=12, fontweight="bold")
ax_acc.set_xlabel("Epoch", fontsize=11)
ax_acc.set_ylabel("Accuracy (%)", fontsize=11)
ax_acc.set_xticks(epochs_list)
ax_acc.grid(True, linestyle="--", alpha=0.6)
ax_acc.legend(fontsize=11)

plt.tight_layout()
curves_save_path = Path(figures_dir) / "resnet18_baseline_curves.png"
fig.savefig(curves_save_path, dpi=150)
plt.show()
print(f"Đã lưu biểu đồ học tập vào: {curves_save_path}")

## 7. Đánh Giá Trên Tập Validation & Chẩn Đoán Lỗi (`labs.lab2.evaluate`)
**Mục đích:**
- Nạp lại checkpoint tốt nhất bằng `load_checkpoint` để đánh giá độc lập trong chế độ `eval()` (không tính gradient).
- `evaluate(model, val_loader, device)`: Tính toán độ mất mát trung bình (`avg_loss`) và độ chính xác tổng thể (`accuracy`).
- `confusion_matrix(model, val_loader, device, num_classes=10)`: Tính ma trận nhầm lẫn kích thước 10x10, trong đó hàng đại diện cho nhãn thực tế và cột đại diện cho nhãn mô hình dự đoán.
- Trực quan hóa ma trận nhầm lẫn bằng heatmap có chú thích số lượng mẫu.

In [ ]:
# 1. Khởi tạo cấu trúc mô hình và nạp lại trọng số tốt nhất từ checkpoint
best_eval_model = build_model("resnet18", num_classes=10, freeze_backbone=True, pretrained=False)
best_eval_model = load_checkpoint(best_eval_model, baseline_ckpt, device=device)
print(f"Nạp thành công mô hình tối ưu từ checkpoint: {baseline_ckpt}")

# 2. Đánh giá độ mất mát và độ chính xác trên tập Validation
val_loss, val_acc = evaluate(best_eval_model, val_loader, device=device)
print("\n--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VALIDATION ---")
print(f"+ Validation Loss    : {val_loss:.4f}")
print(f"+ Validation Accuracy: {val_acc * 100:.2f}% ({int(val_acc * len(val_idx))}/{len(val_idx)} mẫu chính xác)")

# 3. Tính toán Ma trận nhầm lẫn (Confusion Matrix)
cm = confusion_matrix(best_eval_model, val_loader, device=device, num_classes=10)
print(f"Ma trận nhầm lẫn thu được có kích thước: {cm.shape}")

In [ ]:
# Vẽ Heatmap cho Ma trận nhầm lẫn (Confusion Matrix Heatmap)
fig, ax = plt.subplots(figsize=(9, 8))
cax = ax.matshow(cm, cmap=plt.cm.Blues, alpha=0.85)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="left", fontsize=10)
ax.set_yticklabels(CIFAR10_CLASSES, fontsize=10)
ax.set_xlabel("Nhãn Dự Đoán (Predicted Label)", fontsize=11, fontweight="bold")
ax.set_ylabel("Nhãn Thực Tế (True Label)", fontsize=11, fontweight="bold")
ax.set_title("Ma Trận Nhầm Lẫn Trên Tập Validation (CIFAR-10)", fontsize=13, fontweight="bold", pad=20)

# Ghi rõ số lượng mẫu dự đoán vào từng ô trong ma trận
for i in range(10):
    for j in range(10):
        text_color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), va="center", ha="center", color=text_color, fontsize=9)

fig.colorbar(cax, fraction=0.046, pad=0.04)
plt.tight_layout()
cm_fig_path = Path(figures_dir) / "val_confusion_matrix_baseline.png"
fig.savefig(cm_fig_path, dpi=150)
plt.show()
print(f"Đã lưu biểu đồ ma trận nhầm lẫn vào: {cm_fig_path}")

## 8. Phân Tích Lỗi & Dự Đoán Mẫu (`labs.lab2.evaluate`)
**Mục đích:**
- `top_confusions(cm, class_names, k=5)`: Lọc ra 5 cặp lớp mà mô hình nhầm lẫn nhiều nhất (ví dụ: `cat` bị đoán thành `dog`, `automobile` bị đoán thành `truck`).
- Trực quan hóa 8 ảnh mẫu dự đoán từ tập validation kèm nhãn thực tế và nhãn dự đoán để quan sát các trường hợp đoán đúng (màu xanh) và đoán sai (màu đỏ).

In [ ]:
# 1. Tìm và hiển thị Top 5 cặp nhầm lẫn thường gặp nhất
top_err_pairs = top_confusions(cm, list(CIFAR10_CLASSES), k=5)

print("=== TOP 5 CẶP LỚP DỄ NHẦM LẪN NHẤT TRÊN TẬP VALIDATION ===")
for rank, (true_c, pred_c, count) in enumerate(top_err_pairs, 1):
    print(f"  {rank}. Lớp thật: '{true_c:12s}' -> Bị đoán sai thành: '{pred_c:12s}' | Số lần nhầm: {count:3d} mẫu")

# 2. Trực quan hóa 8 ảnh mẫu dự đoán thực tế
best_eval_model.eval()
val_iter = iter(val_loader)
sample_val_imgs, sample_val_targets = next(val_iter)

with torch.no_grad():
    sample_outputs = best_eval_model(sample_val_imgs.to(device))
    sample_predictions = sample_outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

# Đảo ngược chuẩn hóa ImageNet để hiển thị màu sắc ảnh thực tế
mean_arr = np.array(IMAGENET_MEAN)
std_arr = np.array(IMAGENET_STD)

for idx in range(8):
    img_display = sample_val_imgs[idx].permute(1, 2, 0).numpy()
    img_display = np.clip(img_display * std_arr + mean_arr, 0.0, 1.0)

    true_name = CIFAR10_CLASSES[sample_val_targets[idx].item()]
    pred_name = CIFAR10_CLASSES[sample_predictions[idx].item()]
    is_correct = (true_name == pred_name)

    ax = axes[idx]
    ax.imshow(img_display)
    ax.axis("off")
    title_color = "#2ca02c" if is_correct else "#d62728"
    status_text = "ĐÚNG" if is_correct else "SAI"
    ax.set_title(f"Thật: {true_name}\nĐoán: {pred_name} [{status_text}]", color=title_color, fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 9. Thử Nghiệm So Sánh Đa Mô Hình & Siêu Tham Số (`labs.lab2.experiment`)
**Mục đích:**
- `run_comparison`: Tự động huấn luyện và đánh giá có kiểm soát trên nhiều mô hình và cấu hình siêu tham số:
  1. So sánh kiến trúc: `resnet18` vs `densenet121`.
  2. So sánh chiến lược học: Đóng băng hoàn toàn (`freeze_backbone=True`) vs Fine-tuning mở khối cuối (`unfreeze_blocks=1`).
  3. So sánh tốc độ học (Learning Rate): `lr=1e-3` vs `lr=1e-4`.
- `save_results`: Lưu bảng kết quả so sánh dưới dạng file CSV tại `artifacts/runs/model_comparison.csv`.
- Đọc kết quả bằng `pandas` và vẽ biểu đồ so sánh trực quan về cả độ chính xác (`best_val_acc`) lẫn thời gian huấn luyện (`train_seconds`).

> **Lưu ý về Sai số thống kê (Noise & Standard Error):**
> Với $n = 5,000$ ảnh trong tập validation, sai số chuẩn của độ chính xác là $SE = \sqrt{p(1-p)/n}$. Khi $p \approx 0.85$, $SE \approx \sqrt{0.85 \times 0.15 / 5000} \approx 0.5\%$. Khi so sánh các cấu hình chỉ với 1 hạt giống ngẫu nhiên (seed), mức chênh lệch nhỏ hơn khoảng 2 lần sai số chuẩn ($< 1\%$) chưa đủ cơ sở thống kê để kết luận cấu hình này vượt trội hơn cấu hình kia.

In [ ]:
# Thiết lập thư mục lưu trữ các lần chạy thử nghiệm
runs_dir = str(artifact_dir("lab2", "runs"))

# 1. Danh sách các mô hình đưa vào so sánh
models_to_compare = ["resnet18", "densenet121"]

# 2. Lưới siêu tham số thử nghiệm (Hyperparameter Grid)
comparison_grid = [
    {
        "lr": 1e-3,
        "epochs": 2,
        "optimizer_name": "adam",
        "freeze_backbone": True,
        "unfreeze_blocks": 0,
        "weight_decay": 1e-4
    },
    {
        "lr": 1e-4,
        "epochs": 2,
        "optimizer_name": "adam",
        "freeze_backbone": True,
        "unfreeze_blocks": 1,  # Fine-tune khối cuối
        "weight_decay": 1e-4
    },
]

print(f"[Bắt đầu Thử nghiệm So sánh trên {len(models_to_compare)} kiến trúc và {len(comparison_grid)} bộ siêu tham số]")
exp_results = run_comparison(
    model_names=models_to_compare,
    hparam_grid=comparison_grid,
    loaders=(train_loader, val_loader, test_loader),
    device=device,
    log_root=runs_dir
)

# 3. Lưu bảng so sánh dạng file CSV
csv_out_path = str(Path(runs_dir) / "model_comparison.csv")
save_results(exp_results, csv_out_path)
print(f"\nĐã lưu thành công kết quả so sánh vào: {csv_out_path}")

In [ ]:
# Đọc và hiển thị bảng tổng hợp kết quả bằng Pandas
df_comparison = pd.read_csv(csv_out_path)
print("BẢNG KẾT QUẢ THỬ NGHIỆM SO SÁNH (MODEL COMPARISON TABLE):")
display(df_comparison)

# Vẽ biểu đồ so sánh nếu có dữ liệu
if not df_comparison.empty and "best_val_acc" in df_comparison.columns:
    fig, (ax_acc_cmp, ax_time_cmp) = plt.subplots(1, 2, figsize=(14, 5))

    labels = [
        f"{row['model']}\nlr={row.get('hp_lr','')}\nunfreeze={row.get('hp_unfreeze_blocks', 0)}"
        for _, row in df_comparison.iterrows()
    ]

    # Biểu đồ so sánh Độ chính xác Validation tốt nhất
    bars_acc = ax_acc_cmp.bar(range(len(df_comparison)), df_comparison["best_val_acc"] * 100, color="#3470a3", width=0.5)
    ax_acc_cmp.set_xticks(range(len(df_comparison)))
    ax_acc_cmp.set_xticklabels(labels, fontsize=9)
    ax_acc_cmp.set_ylabel("Độ chính xác Validation (%) ", fontsize=11, fontweight="bold")
    ax_acc_cmp.set_title("So sánh Độ chính xác giữa các Cấu hình", fontsize=12, fontweight="bold")
    ax_acc_cmp.grid(axis="y", linestyle="--", alpha=0.6)
    for b in bars_acc:
        h = b.get_height()
        ax_acc_cmp.text(b.get_x() + b.get_width()/2.0, h + 0.5, f"{h:.2f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

    # Biểu đồ so sánh Thời gian huấn luyện
    bars_time = ax_time_cmp.bar(range(len(df_comparison)), df_comparison["train_seconds"], color="#e26d5c", width=0.5)
    ax_time_cmp.set_xticks(range(len(df_comparison)))
    ax_time_cmp.set_xticklabels(labels, fontsize=9)
    ax_time_cmp.set_ylabel("Thời gian huấn luyện (giây)", fontsize=11, fontweight="bold")
    ax_time_cmp.set_title("So sánh Thời gian Huấn luyện (giây)", fontsize=12, fontweight="bold")
    ax_time_cmp.grid(axis="y", linestyle="--", alpha=0.6)
    for b in bars_time:
        h = b.get_height()
        ax_time_cmp.text(b.get_x() + b.get_width()/2.0, h + 0.5, f"{h:.1f}s", ha="center", va="bottom", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 10. Đánh Giá Cuối Cùng Trên Tập Test Độc Lập (`labs.lab2.evaluate`)
**Quy tắc học máy quan trọng:**
- Mọi lựa chọn về cấu trúc mạng, phương pháp đóng băng/fine-tuning, tốc độ học, số epoch và trọng số suy giảm đều được đưa ra dựa trên kết quả của tập **Validation** ở các bước trước.
- Tập **Test** (10,000 ảnh) hoàn toàn độc lập và **chỉ được gọi duy nhất một lần** cho cấu hình chiến thắng để báo cáo năng lực tổng quát hóa khách quan của mô hình, tuyệt đối không dùng để chỉnh sửa siêu tham số.

In [ ]:
# Đánh giá mô hình tốt nhất trên tập Test độc lập
print("[Đánh giá Cuối cùng] Đang thực thi đánh giá trên 10,000 mẫu của Test Loader...")
final_test_loss, final_test_acc = evaluate(best_eval_model, test_loader, device=device)

print("=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (FINAL TEST RESULTS):")
print(f"+ Test Loss        : {final_test_loss:.4f}")
print(f"+ Test Accuracy    : {final_test_acc * 100:.2f}% ({int(final_test_acc * len(test_set_raw)):,}/{len(test_set_raw):,} mẫu đúng)")
print(f"+ Validation Acc   : {val_acc * 100:.2f}%")
gap = abs(val_acc - final_test_acc) * 100
print(f"+ Độ lệch Val-Test : {gap:.2f}%")
if gap < 2.0:
    print("+ Nhận xét: Độ lệch giữa Val và Test rất nhỏ (< 2%), mô hình có tính tổng quát hóa cao và không bị overfitting.")
else:
    print("+ Nhận xét: Có sự chênh lệch nhất định giữa Val và Test, cần theo dõi thêm regularization.")
print("=" * 60)

## 11. Tổng Kết & Chuẩn Bị Câu Hỏi Vấn Đáp Bảo Vệ (Oral Exam Defense)

### Tóm Tắt Kết Quả Chính (Summary):
1. **Hiệu quả của Transfer Learning:** Việc sử dụng backbone CNN pre-trained trên ImageNet giúp mô hình nhanh chóng hội tụ và đạt độ chính xác cao trên CIFAR-10 ngay cả khi kích thước ảnh gốc nhỏ (32x32).
2. **So sánh Đóng băng vs Fine-tuning:** 
   - Khi đóng băng hoàn toàn backbone (`freeze_backbone=True`), số tham số cần học chỉ chiếm khoảng $0.05\%$ tổng tham số của ResNet-18 (5,130 / 11,181,642 tham số), giúp thời gian huấn luyện mỗi epoch rất nhanh và ngăn chặn hiện tượng phá hủy các đặc trưng tổng quát đã học từ ImageNet.
   - Khi mở khóa khối tích chập bậc cao (`layer4` với `unfreeze_blocks=1`), mô hình học được các đặc trưng ngữ nghĩa sát với miền ảnh CIFAR-10 hơn, giúp tăng nhẹ độ chính xác nhưng cần tốc độ học nhỏ (`lr=1e-4`) để tránh phân kỳ.
3. **Chuẩn hóa dữ liệu:** Chuẩn hóa ImageNet (`IMAGENET_MEAN`, `IMAGENET_STD`) là thành phần thiết yếu; việc không chuẩn hóa hoặc chuẩn hóa sai dải giá trị sẽ khiến các bộ lọc tích chập pre-trained hoạt động sai lệch.

---

### 5 Câu Hỏi Vấn Đáp Trọng Tâm Cùng Câu Trả Lời Chuẩn Mực:

**Câu 1: Tại sao phải thay lớp cuối cùng của mô hình pre-trained và cách tính số tham số cần huấn luyện?**
- *Trả lời:* Mô hình pre-trained từ ImageNet ban đầu có lớp phân loại đầu ra gồm 1,000 lớp. Bài toán CIFAR-10 chỉ có 10 lớp, nên ta phải thay lớp fully-connected (`fc` của ResNet hoặc `classifier` của DenseNet) bằng lớp `nn.Linear(in_features, 10)` mới. Lớp mới này mặc định có `requires_grad=True`. Khi đóng băng backbone, số tham số huấn luyện chỉ bằng số tham số của lớp này: $(in\_features \times 10) + 10$ (ví dụ: với ResNet-18 là $512 \times 10 + 10 = 5,130$ tham số).

**Câu 2: Sự khác nhau bản chất giữa Feature Extraction và Fine-Tuning? Khi nào nên chọn phương pháp nào?**
- *Trả lời:*
  - *Feature Extraction:* Cố định toàn bộ trọng số của backbone (`requires_grad=False`), chỉ xem mạng như một bộ trích xuất đặc trưng cố định và chỉ học lớp phân loại cuối. Phù hợp khi tập dữ liệu mục tiêu nhỏ, tương đồng với tập dữ liệu gốc (ImageNet) hoặc tài nguyên tính toán (CPU/GPU) hạn chế.
  - *Fine-Tuning:* Mở khóa một phần hoặc toàn bộ các tầng cuối của backbone để cập nhật tiếp trọng số cùng với lớp phân loại. Phù hợp khi có tập dữ liệu mới đủ lớn hoặc có sự khác biệt về phong cách ảnh so với ImageNet. Cần dùng learning rate nhỏ hơn bình thường (khoảng $10^{-4}$ hoặc $10^{-5}$) để không làm phá vỡ các trọng số pre-trained tốt.

**Câu 3: Tại sao bắt buộc phải chuẩn hóa ảnh bằng `IMAGENET_MEAN` và `IMAGENET_STD` thay vì chia 255 đơn thuần?**
- *Trả lời:* Các trọng số tích chập trong mô hình pre-trained của torchvision đã được tối ưu hóa trong quá trình huấn luyện trên ImageNet với phân phối đầu vào có kỳ vọng 0 và phương sai 1 theo bộ mean `[0.485, 0.456, 0.406]` và std `[0.229, 0.224, 0.225]`. Nếu ta chỉ chia cho 255 (dải $[0, 1]$) mà không chuẩn hóa, phân phối đầu vào sẽ bị lệch (mean $\approx 0.5$ thay vì 0), làm các phép kích hoạt phi tuyến (ReLU) và chuẩn hóa theo lô (BatchNorm) kích hoạt lệch miền, dẫn đến giảm mạnh độ chính xác.

**Câu 4: Tại sao chỉ áp dụng Data Augmentation trên tập Train mà không áp dụng trên tập Validation và Test?**
- *Trả lời:* Data Augmentation (như xoay ảnh, cắt ngẫu nhiên, lật ngang) mang tính ngẫu nhiên, mục đích là mở rộng không gian mẫu và tăng cường tính bất biến (invariance) cho mô hình trong quá trình học. Ngược lại, tập Validation và Test được dùng để đo lường năng lực thực tế một cách chuẩn xác, nhất quán và có thể tái lập. Nếu áp dụng biến đổi ngẫu nhiên trên tập đánh giá, thước đo accuracy sẽ bị dao động ngẫu nhiên theo từng lần chạy, làm sai lệch việc so sánh mô hình.

**Câu 5: Dựa vào Confusion Matrix, làm thế nào để giải thích các cặp lớp bị nhầm lẫn nhiều nhất và hướng cải thiện là gì?**
- *Trả lời:* Các cặp lớp nhầm lẫn nhiều nhất trong CIFAR-10 thường là các lớp cùng họ hình thái (ví dụ: động vật 4 chân như `cat` vs `dog`, hoặc phương tiện giao thông như `automobile` vs `truck`). Khi kích thước ảnh bị thu nhỏ hoặc bị biến dạng góc nhìn, các đặc trưng cục bộ (tai, đuôi, bánh xe) trở nên mờ nhạt. Hướng cải thiện: Sử dụng kích thước ảnh lớn hơn (224x224), bổ sung kỹ thuật tăng cường dữ liệu đặc thù (CutMix, AutoAugment), hoặc fine-tune sâu hơn vào các tầng tích chập trung gian.